# Modulo 7 · Lezione 9 — sklearn Pipelines e Distribuzione dei Modelli

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Perché usare le Pipeline |
| 2 | `Pipeline` di base |
| 3 | `ColumnTransformer` + Pipeline completa |
| 4 | Pipeline + GridSearchCV |
| 5 | Salvare e caricare modelli con `joblib` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
)
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
print("Librerie pronte")

---
## 1. Perché usare le Pipeline

```
Senza Pipeline (codice fragile e incline al data leakage):

  scaler = StandardScaler()
  X_tr_sc = scaler.fit_transform(X_tr)   # OK: apprende i parametri sul train
  X_te_sc = scaler.transform(X_te)       # OK: applica gli stessi
  modelo.fit(X_tr_sc, y_tr)
  modelo.predict(X_te_sc)
  # Problema: facendo CV, devi ripetere questo 5 volte manualmente
  # Errore frequente: fare fit_transform(X_completo) prima di dividere → LEAKAGE

Con Pipeline:

  pipe = Pipeline([('scaler', StandardScaler()), ('clf', RandomForest())])
  pipe.fit(X_tr, y_tr)       # scaler.fit_transform(X_tr) + clf.fit
  pipe.predict(X_te)         # scaler.transform(X_te) + clf.predict
  cross_val_score(pipe, X, y, cv=5)  # garantisce NESSUN leakage in ogni fold
  GridSearchCV(pipe, param_grid)     # cerca gli iperparametri SENZA leakage

Vantaggi della Pipeline:
  1. Garantisce l'ordine corretto dei passi
  2. Previene il data leakage in CV e GridSearch
  3. Salva TUTTA la pipeline (scaler + modello) in un unico file
  4. Una sola riga .predict() in produzione
```

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=200, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Modo SCORRETTO (leakage nella CV)
scaler_mal = StandardScaler()
X_mal = scaler_mal.fit_transform(X)   # apprende media/std di TUTTO (include il test)
scores_mal = cross_val_score(
    LogisticRegression(max_iter=500),
    X_mal, y, cv=5, scoring='accuracy'
)

# Modo CORRETTO (pipeline)
pipe_bien = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=500))
])
scores_bien = cross_val_score(pipe_bien, X, y, cv=5, scoring='accuracy')

print("Confronto leakage vs Pipeline:")
print(f"  Con leakage (score gonfiato): {scores_mal.mean():.4f} ± {scores_mal.std():.4f}")
print(f"  Senza leakage (Pipeline):     {scores_bien.mean():.4f} ± {scores_bien.std():.4f}")
print(f"  Differenza: {scores_mal.mean() - scores_bien.mean():.4f} punti di distorsione")

---
## 2. `Pipeline` di base

In [ ]:
# Anatomia di una Pipeline
pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # passo 1: imputare i NaN
    ('scaler',  StandardScaler()),                    # passo 2: scalare
    ('clf',     LogisticRegression(max_iter=1000)),   # passo finale: modello
])

print("Passi della pipeline:")
for nombre, paso in pipe.steps:
    print(f"  [{nombre}] {type(paso).__name__}")

print(f"\nUltimo estimatore: {pipe[-1]}")

# Dataset con valori NaN per la dimostrazione
X_nan = X.copy()
mascara = np.random.rand(*X_nan.shape) < 0.1  # 10% di NaN a caso
X_nan[mascara] = np.nan
X_tr_nan, X_te_nan, y_tr_n, y_te_n = train_test_split(X_nan, y, test_size=0.2, random_state=42)

pipe.fit(X_tr_nan, y_tr_n)
acc = accuracy_score(y_te_n, pipe.predict(X_te_nan))
print(f"\nAccuracy sul test (dati con NaN): {acc:.4f}")

# Accedere ai passi interni
imputer_ajustado = pipe.named_steps['imputer']
print(f"\nMediane apprese (prime 5): {imputer_ajustado.statistics_[:5].round(3)}")

---
## 3. `ColumnTransformer` + Pipeline completa

```
ColumnTransformer permette di applicare trasformazioni DIVERSE a colonne diverse:

  Colonne numeriche → imputare (mediana) + scalare
  Colonne ordinali  → imputare (più_frequente) + OrdinalEncoder
  Colonne nominali  → imputare (più_frequente) + OneHotEncoder

  Tutto in parallelo, senza data leakage, dentro la Pipeline.
```

In [ ]:
# Dataset realistico con tipi misti
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'edad':       np.random.randint(18, 70, n).astype(float),
    'salario':    np.random.exponential(30000, n) + 15000,
    'antiguedad': np.random.randint(0, 30, n).astype(float),
    'nivel':      np.random.choice(['junior','mid','senior'], n),
    'ciudad':     np.random.choice(['Madrid','Barcelona','Valencia','Bilbao'], n),
    'departamento': np.random.choice(['Tech','Sales','HR','Finance'], n),
})
# Introdurre NaN
for col in ['edad','salario','nivel','ciudad']:
    idx = np.random.choice(n, 20, replace=False)
    df.loc[idx, col] = np.nan

# Target: promosso o no
y_df = ((df['antiguedad'].fillna(df['antiguedad'].median()) > 5) &
        (df['salario'].fillna(df['salario'].median()) > 30000)).astype(int)

print("Dataset (prime righe):")
print(df.head())
print(f"\nNaN per colonna:\n{df.isnull().sum()}")

In [ ]:
# Separare le feature per tipo
cols_numericas  = ['edad', 'salario', 'antiguedad']
cols_ordinales  = ['nivel']
cols_nominales  = ['ciudad', 'departamento']

X_df = df[cols_numericas + cols_ordinales + cols_nominales]
X_tr_df, X_te_df, y_tr_df, y_te_df = train_test_split(
    X_df, y_df, test_size=0.2, stratify=y_df, random_state=42
)

# Preprocessore per tipo di colonna
prepro_numerico = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

prepro_ordinal = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[['junior','mid','senior']],
        handle_unknown='use_encoded_value', unknown_value=-1
    )),
])

prepro_nominal = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')),
])

# ColumnTransformer
preprocesador = ColumnTransformer(transformers=[
    ('num', prepro_numerico, cols_numericas),
    ('ord', prepro_ordinal,  cols_ordinales),
    ('nom', prepro_nominal,  cols_nominales),
])

# Pipeline finale: preprocessore + modello
pipeline_completo = Pipeline([
    ('prepro', preprocesador),
    ('clf',    RandomForestClassifier(100, random_state=42, n_jobs=-1)),
])

pipeline_completo.fit(X_tr_df, y_tr_df)
y_pred = pipeline_completo.predict(X_te_df)

print("Pipeline completa (ColumnTransformer + RandomForest):")
print(f"Accuracy sul test: {accuracy_score(y_te_df, y_pred):.4f}")
print(f"\n{classification_report(y_te_df, y_pred, target_names=['Non promosso','Promosso'])}")

In [ ]:
# Ispezionare la forma dei dati trasformati
X_transformado = pipeline_completo.named_steps['prepro'].transform(X_te_df)
print(f"Shape originale X_te: {X_te_df.shape}")
print(f"Shape trasformato:    {X_transformado.shape}")

# Ottenere i nomi delle feature dopo OHE
ohe = pipeline_completo.named_steps['prepro'].named_transformers_['nom'].named_steps['encoder']
nombres_ohe = ohe.get_feature_names_out(cols_nominales)

todos_nombres = cols_numericas + cols_ordinales + list(nombres_ohe)
print(f"\nFeature dopo la trasformazione ({len(todos_nombres)}):")
print(todos_nombres)

# Feature importance
rf = pipeline_completo.named_steps['clf']
importancias = pd.Series(rf.feature_importances_, index=todos_nombres).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
importancias.head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Importanza')
ax.set_title('Top 10 feature più importanti')
plt.tight_layout(); plt.show()

---
## 4. Pipeline + GridSearchCV

In [ ]:
# Cercare gli iperparametri del preprocessore E del modello
# Sintassi: 'nomePasso__nomePasso__iperparametro' (doppio __ per passi annidati)

rejilla = {
    # Iperparametri del modello
    'clf__n_estimators':        [50, 100, 200],
    'clf__max_depth':           [5, None],
    'clf__min_samples_split':   [2, 5],
    # Iperparametro del preprocessore (lo scaler numerico)
    'prepro__num__scaler':      [StandardScaler(), MinMaxScaler()],
}

gs = GridSearchCV(
    pipeline_completo,
    rejilla,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='accuracy',
    refit=True, n_jobs=-1, verbose=1
)
gs.fit(X_tr_df, y_tr_df)

print(f"\nMigliori parametri: {gs.best_params_}")
print(f"Score CV: {gs.best_score_:.4f}")
print(f"Accuracy test: {accuracy_score(y_te_df, gs.predict(X_te_df)):.4f}")

---
## 5. Salvare e caricare modelli con `joblib`

```
Perché salvare il modello:
  - Riallenare è costoso (tempo, GPU, dati)
  - Il server di produzione non esegue il training
  - Versionare i modelli: sapere quale modello ha generato quale predizione

joblib vs pickle:
  joblib è più efficiente con grandi array numpy (interni di sklearn)
  pickle per piccoli oggetti Python
  In sklearn: usa sempre joblib

IMPORTANTE: salva la PIPELINE completa (preprocessore + modello)
  Al caricarla, ti basta .predict(X_raw) → la pipeline fa tutto il preprocessing.
```

In [ ]:
# 1. Salvare la pipeline completa
ruta_modelo = 'modelo_promovidos_v1.joblib'
joblib.dump(gs.best_estimator_, ruta_modelo)
print(f"Modello salvato in: {ruta_modelo}")
print(f"Dimensione del file: {os.path.getsize(ruta_modelo) / 1024:.1f} KB")

# 2. Caricare il modello
modelo_cargado = joblib.load(ruta_modelo)
print(f"\nModello caricato: {type(modelo_cargado).__name__}")

# 3. Usare il modello caricato (dati RAW non preprocessati)
acc_cargado = accuracy_score(y_te_df, modelo_cargado.predict(X_te_df))
print(f"Accuracy del modello caricato: {acc_cargado:.4f}")

# 4. Simulare dati di produzione (un cliente nuovo, con possibili NaN)
cliente_nuevo = pd.DataFrame({
    'edad':           [32],
    'salario':        [45000],
    'antiguedad':     [7],
    'nivel':          ['mid'],
    'ciudad':         ['Madrid'],
    'departamento':   ['Tech'],
})

prediccion = modelo_cargado.predict(cliente_nuevo)[0]
proba      = modelo_cargado.predict_proba(cliente_nuevo)[0]
print(f"\nCliente nuovo:")
print(cliente_nuevo.to_string(index=False))
print(f"\nPredizione: {'Promosso' if prediccion==1 else 'Non promosso'}")
print(f"Probabilità: {proba[1]:.4f} di essere promosso")

In [ ]:
# Buone pratiche: salvare i metadati insieme al modello
import json

metadatos = {
    'version':          'v1.0.0',
    'features_entrada': cols_numericas + cols_ordinales + cols_nominales,
    'target':           'promovido (0=No, 1=Si)',
    'algoritmo':        'RandomForestClassifier',
    'mejores_params':   {k: str(v) for k, v in gs.best_params_.items()},
    'score_cv':         round(gs.best_score_, 4),
    'accuracy_test':    round(acc_cargado, 4),
    'n_train':          len(X_tr_df),
}

ruta_meta = 'modelo_promovidos_v1_meta.json'
with open(ruta_meta, 'w', encoding='utf-8') as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)

print("Metadati salvati:")
print(json.dumps(metadatos, indent=2, ensure_ascii=False))

---
## Pratica

### Esercizio 1 — Pipeline con Titanic

**Consegna:** Usa il dataset Titanic di seaborn:

```python
import seaborn as sns
titanic = sns.load_dataset('titanic')
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'
```

1. Costruisci una Pipeline completa con `ColumnTransformer`:
   - Numeriche: `age`, `sibsp`, `parch`, `fare` → imputare (mediana) + StandardScaler
   - Ordinali: `pclass` → OrdinalEncoder
   - Nominali: `sex`, `embarked` → imputare (più_frequente) + OneHotEncoder
2. Aggiungi un RandomForestClassifier come ultimo passo
3. Valuta con 5-Fold CV
4. Mostra accuracy, F1 e le 5 feature più importanti

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

titanic = sns.load_dataset('titanic').dropna(subset=['survived'])
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
X = titanic[features]; y = titanic['survived']

cols_num  = ['age','sibsp','parch','fare']
cols_ord  = ['pclass']
cols_nom  = ['sex','embarked']

prepro_num = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
prepro_ord = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
prepro_nom = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')),
])

ct = ColumnTransformer([
    ('num', prepro_num, cols_num),
    ('ord', prepro_ord, cols_ord),
    ('nom', prepro_nom, cols_nom),
])

pipe = Pipeline([
    ('prepro', ct),
    ('clf',    RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1)),
])

cv = StratifiedKFold(5, shuffle=True, random_state=42)
res = cross_validate(pipe, X, y, cv=cv, scoring={'acc':'accuracy','f1':'f1'}, return_train_score=True)
print(f"Accuracy CV: {res['test_acc'].mean():.4f} ± {res['test_acc'].std():.4f}")
print(f"F1 CV:       {res['test_f1'].mean():.4f} ± {res['test_f1'].std():.4f}")

# Feature importances
pipe.fit(X, y)
rf = pipe.named_steps['clf']
ohe_names = pipe.named_steps['prepro'].named_transformers_['nom'].named_steps['ohe'].get_feature_names_out(cols_nom)
nombres = cols_num + cols_ord + list(ohe_names)
imp = pd.Series(rf.feature_importances_, index=nombres).sort_values(ascending=False)
print(f"\nTop 5 feature: {imp.head(5).index.tolist()}")
```
</details>

---
### Esercizio 2 — Confrontare 3 modelli nella stessa Pipeline

**Consegna:** Usa il dataset diabetes:

```python
from sklearn.datasets import load_diabetes
X, y = load_diabetes(return_X_y=True)
# Binarizzare il target (sopra o sotto la mediana)
y = (y > np.median(y)).astype(int)
```

1. Definisci una Pipeline base: `StandardScaler → clf`
2. Prova 3 classificatori distinti con cross_val_score:
   - LogisticRegression
   - RandomForestClassifier(100)
   - GradientBoostingClassifier(100)
3. Mostra una tabella comparativa con accuracy e F1
4. Quale sceglieresti e perché?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

X, y = load_diabetes(return_X_y=True)
y = (y > np.median(y)).astype(int)

cv = StratifiedKFold(5, shuffle=True, random_state=42)
filas = []
for nombre, clf in [
    ('LogisticRegression', LogisticRegression(max_iter=5000, random_state=42)),
    ('RandomForest',       RandomForestClassifier(100, random_state=42, n_jobs=-1)),
    ('GradientBoosting',   GradientBoostingClassifier(100, random_state=42)),
]:
    pipe = Pipeline([('sc', StandardScaler()), ('clf', clf)])
    res = cross_validate(pipe, X, y, cv=cv, scoring={'acc':'accuracy','f1':'f1'}, n_jobs=-1)
    filas.append({
        'Modelo':   nombre,
        'Acc_media': res['test_acc'].mean(),
        'Acc_std':   res['test_acc'].std(),
        'F1_media':  res['test_f1'].mean(),
        'F1_std':    res['test_f1'].std(),
    })

df_comp = pd.DataFrame(filas).sort_values('F1_media', ascending=False)
print(df_comp.round(4).to_string(index=False))
print("\nScelta suggerita: GradientBoosting di solito vince su dataset tabulari medi.")
print("Per dataset grandi, RandomForest può essere più veloce.")
```
</details>

---
### Esercizio 3 — GridSearch completo

**Consegna:** Usa la Pipeline dell'Esercizio 1 (Titanic) e ottimizzane gli iperparametri:

1. Definisci una griglia che esplori:
   - `clf__n_estimators`: [50, 100, 200]
   - `clf__max_depth`: [5, 10, None]
   - `clf__class_weight`: ['balanced', None]
2. Usa `GridSearchCV` con 5 fold
3. Confronta il modello di default vs quello ottimizzato
4. Di quanto è migliorato l'F1?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import seaborn as sns
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
# Riutilizzare pipe e i dati dell'esercizio 1

titanic = sns.load_dataset('titanic').dropna(subset=['survived'])
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
X, y = titanic[features], titanic['survived']

# (riutilizzare il pipe dell'esercizio 1...)
# Assicurati di avere pipe definito prima di eseguire questo

rejilla = {
    'clf__n_estimators':  [50, 100, 200],
    'clf__max_depth':     [5, 10, None],
    'clf__class_weight':  ['balanced', None],
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)

# Baseline
f1_base = cross_val_score(pipe, X, y, cv=cv, scoring='f1').mean()

# GridSearch
gs = GridSearchCV(pipe, rejilla, cv=cv, scoring='f1', refit=True, n_jobs=-1)
gs.fit(X, y)

print(f"F1 baseline (di default): {f1_base:.4f}")
print(f"F1 ottimizzato (CV):      {gs.best_score_:.4f}")
print(f"Miglioramento: +{gs.best_score_ - f1_base:.4f}")
print(f"Migliori parametri: {gs.best_params_}")
```
</details>

---
### Esercizio 4 — Ciclo completo: salvare e caricare

**Consegna:** Simula il ciclo completo di produzione:

1. Allena la pipeline dell'Esercizio 1 sull'80% dei dati
2. Salvala con `joblib.dump` come `'titanic_modelo.joblib'`
3. Salva i suoi metadati in `'titanic_modelo_meta.json'`
4. Ricaricala con `joblib.load`
5. Genera predizioni per 3 passeggeri fittizi:
   - Passeggero A: pclass=1, sex='female', age=25, sibsp=0, parch=0, fare=200, embarked='C'
   - Passeggero B: pclass=3, sex='male', age=35, sibsp=0, parch=0, fare=7, embarked='S'
   - Passeggero C: pclass=2, sex='male', age=10, sibsp=1, parch=2, fare=30, embarked='Q'
6. Mostra la predizione e la probabilità di sopravvivenza

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import joblib, json
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

titanic = sns.load_dataset('titanic').dropna(subset=['survived'])
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
X, y = titanic[features], titanic['survived']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Usare il pipe dell'esercizio 1 già definito
pipe.fit(X_tr, y_tr)
acc_te = accuracy_score(y_te, pipe.predict(X_te))

# Salvare
joblib.dump(pipe, 'titanic_modelo.joblib')
meta = {
    'features': features,
    'target': 'survived (0=No, 1=Si)',
    'accuracy_test': round(acc_te, 4),
    'n_train': len(X_tr),
}
with open('titanic_modelo_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print("Modello salvato.")

# Caricare e predire
modelo = joblib.load('titanic_modelo.joblib')
pasajeros = pd.DataFrame([
    {'pclass':1,'sex':'female','age':25,'sibsp':0,'parch':0,'fare':200,'embarked':'C'},
    {'pclass':3,'sex':'male',  'age':35,'sibsp':0,'parch':0,'fare':7,  'embarked':'S'},
    {'pclass':2,'sex':'male',  'age':10,'sibsp':1,'parch':2,'fare':30, 'embarked':'Q'},
], columns=features)

for i, (_, row) in enumerate(pasajeros.iterrows()):
    X_row = pd.DataFrame([row])
    pred  = modelo.predict(X_row)[0]
    proba = modelo.predict_proba(X_row)[0][1]
    print(f"Passeggero {chr(65+i)}: {'Sopravvissuto' if pred==1 else 'Non sopravvissuto'} "
          f"(prob={proba:.3f})")
```
</details>

---
### Esercizio 5 (sfida) — Fabbrica di Pipeline

**Consegna:** Implementa `crear_pipeline(modelo, cols_num, cols_ord, cols_nom, ord_cats=None)` che:

1. Crea automaticamente il ColumnTransformer per i 3 tipi di colonne
2. Aggiunge il modello alla fine
3. Restituisce la Pipeline pronta all'uso

Provala creando 3 pipeline distinte (RandomForest, LogisticRegression, GradientBoosting) con lo stesso ColumnTransformer e confrontale con `cross_val_score`.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def crear_pipeline(modelo, cols_num=None, cols_ord=None, cols_nom=None, ord_cats=None):
    transformers = []
    if cols_num:
        transformers.append(('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc',  StandardScaler()),
        ]), cols_num))
    if cols_ord:
        cats = ord_cats if ord_cats else 'auto'
        transformers.append(('ord', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder(categories=cats,
                                   handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), cols_ord))
    if cols_nom:
        transformers.append(('nom', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')),
        ]), cols_nom))

    ct = ColumnTransformer(transformers)
    return Pipeline([('prepro', ct), ('clf', modelo)])

# Provare con Titanic
import seaborn as sns
titanic = sns.load_dataset('titanic').dropna(subset=['survived'])
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
X, y = titanic[features], titanic['survived']

cv = StratifiedKFold(5, shuffle=True, random_state=42)
modelos = [
    ('RandomForest',     RandomForestClassifier(100, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('LogisticReg',      LogisticRegression(max_iter=5000, random_state=42)),
    ('GradientBoosting', GradientBoostingClassifier(100, random_state=42)),
]

print(f"{'Modello':22} {'Acc CV':>10} {'F1 CV':>10}")
print("-" * 44)
for nombre, clf in modelos:
    pipe = crear_pipeline(
        clf,
        cols_num=['age','sibsp','parch','fare'],
        cols_ord=['pclass'],
        cols_nom=['sex','embarked'],
    )
    acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy', n_jobs=-1).mean()
    f1  = cross_val_score(pipe, X, y, cv=cv, scoring='f1',       n_jobs=-1).mean()
    print(f"{nombre:22} {acc:>10.4f} {f1:>10.4f}")
```
</details>

---
## Riepilogo

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

# Pipeline di base
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(100)),
])

# ColumnTransformer per tipi misti
ct = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer()), ('sc',StandardScaler())]), cols_num),
    ('ord', OrdinalEncoder(), cols_ord),
    ('nom', OneHotEncoder(sparse_output=False, drop='first'), cols_nom),
])
pipe_full = Pipeline([('prepro', ct), ('clf', RandomForestClassifier())])

# GridSearchCV sulla pipeline
gs = GridSearchCV(pipe_full, {'clf__n_estimators': [50,100]}, cv=5)
gs.fit(X_tr, y_tr)      # accesso: 'nomePasso__iperparametro'

# Salvare / caricare
joblib.dump(gs.best_estimator_, 'modelo.joblib')   # salva TUTTA la pipeline
modelo = joblib.load('modelo.joblib')
pred = modelo.predict(X_nuevo_raw)                  # senza preprocessare manualmente
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 10** metteremo tutto in pratica con un **Progetto finale end-to-end** usando il dataset California Housing: EDA, pipeline completa, modelli multipli, GridSearchCV e report finale.